<a href="https://colab.research.google.com/github/raheelarif86/AI_Training_November25/blob/main/Copy_of_crewai_multiagent_level_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gitex workshop: Create Multi Agents to Research and Write an Article

In this workshop, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

https://huggingface.co/spaces/decodingdatascience/fromprompttopost

In [ ]:
#install packages
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [ ]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.



In [ ]:
#setting the model
import os

os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
#os.environ["OPENAI_MODEL_NAME"] = 'gpt-4.1-nano-2025-04-14'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [ ]:
# Retrieve API key from Colab secrets and set it in environment variables
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('openai')

In [ ]:
from crewai_tools import (
    DirectoryReadTool,
    FileReadTool,
    SerperDevTool,
    WebsiteSearchTool
)

In [ ]:
# Set up API keys - enter from https://serper.dev/billing
os.environ["SERPER_API_KEY"] = "218873db9b73d345d3083e2f1f29cb74b14d4ddb" # serper.dev API key

# Instantiate tools
docs_tool = DirectoryReadTool(directory='./blog-posts')
file_tool = FileReadTool()
search_tool = SerperDevTool()
web_rag_tool = WebsiteSearchTool()

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic."    ,
    tools=[search_tool, web_rag_tool,docs_tool,file_tool],
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic in 1500 words: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.1500 words",
    agent=writer,
)

### Task: Edit

In [ ]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    planning=True,
    verbose=4
)

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

## Running the Crew
## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "GPT 5.2"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on GPT 5.2.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I need to gather information on the latest trends, key players, and noteworthy news on GPT 5.2 to prioritize them in the blog article.

Action: Search the internet
Action Input: {"search_query": "GPT 5.2 latest trends and key players"} 


Search results: Title: Introducing GPT-5.2 - OpenAI
Link: https://openai.com/index/introducing-gpt-5-2/
Snippet: GPT‑5.2 Thinking sets a new state of the art in long-context reasoning, achieving leading performance on OpenAI MRCRv2—an evaluation that tests ...
---
Title: GPT 5.2: Benchmarks, Model Breakdown, and Real-World ...
Link: ht

In [ ]:
from IPython.display import Markdown
Markdown(result)

# Exploring the Advancements of GPT 5.2: A Comprehensive Guide

## Introduction
Artificial Intelligence (AI) has become an integral part of our daily lives, revolutionizing various industries and enhancing efficiency. One of the latest advancements in AI technology is GPT 5.2, developed by OpenAI. This state-of-the-art model has pushed the boundaries of AI capabilities, offering remarkable improvements in long-context reasoning, tool calling capabilities, coding enhancements, and vision improvements.

## Latest Trends in GPT 5.2
### Advancements in long-context reasoning
GPT 5.2 has significantly improved its ability to understand and analyze long-context information, making it more adept at processing complex data and generating more accurate responses.

### Tool calling capabilities
One of the key features of GPT 5.2 is its enhanced tool calling capabilities, allowing users to interact with various applications and services seamlessly, leading to increased productivity and efficiency.

### Coding enhancements
GPT 5.2 has introduced coding enhancements that simplify the coding process, making it easier for developers to write and debug code efficiently.

### Vision improvements
With enhanced vision capabilities, GPT 5.2 can now analyze and interpret visual data more accurately, opening up new possibilities for applications in image recognition and processing.

## Key Players in the GPT 5.2 Ecosystem
OpenAI is at the forefront of AI research and development, leading the way in advancing technologies like GPT 5.2. Comparisons with other AI models like Gemini 3.0 and Opus 4.5 highlight the superiority of GPT 5.2 in terms of performance and capabilities.

## Noteworthy News on GPT 5.2
GPT 5.2 has made significant strides in industry tasks, outperforming professionals in various occupations and showcasing its potential to revolutionize the way we work and interact with AI technologies.

## Target Audience Analysis
### Knowledge workers
Knowledge workers can benefit from GPT 5.2's advanced capabilities in processing and analyzing information, improving decision-making processes and enhancing productivity.

### Educators and students
Educators and students can leverage GPT 5.2 for educational purposes, facilitating learning and knowledge dissemination through tailored content generation and interactive tools.

### Creators
Content creators can explore GPT 5.2's vision capabilities to enhance visual content creation and storytelling, enabling them to engage audiences in new and innovative ways.

### Industry professionals
Industry professionals across various sectors can utilize GPT 5.2 to streamline operations, optimize processes, and drive innovation through its advanced AI capabilities.

## Interests and Pain Points of the Target Audience
### General intelligence
GPT 5.2's general intelligence capabilities cater to the diverse needs of users, providing comprehensive solutions for a wide range of tasks and applications.

### Long-context understanding
The model's improved long-context understanding enhances its ability to process complex information and generate more contextually relevant responses.

### Agentic tool-calling
Users can interact with GPT 5.2 through agentic tool-calling, enabling seamless integration with external applications and services for enhanced functionality.

### Vision capabilities
GPT 5.2's vision capabilities open up new opportunities for visual data analysis and interpretation, expanding its potential applications in image-related tasks.

### Tailored content generation
The model's ability to generate tailored content based on user input and preferences allows for personalized interactions and customized outputs.

## Call to Action
As we delve into the advancements of GPT 5.2, I encourage readers to explore the world of AI and engage with OpenAI's resources and tools to experience the transformative power of cutting-edge AI technologies firsthand.

In conclusion, GPT 5.2 represents a significant leap forward in AI technology, offering unparalleled advancements in long-context reasoning, tool calling capabilities, coding enhancements, and vision improvements. As we witness the impact of AI on various industries and sectors, it is clear that GPT 5.2 is leading the way towards a future where intelligent machines play a crucial role in shaping our world.

## Resources
- OpenAI website
- DataCamp blog
- VentureBeat article
- Forbes publication

Let's embrace the possibilities of AI and unlock the potential of GPT 5.2 for a brighter and more innovative future.

# Deploy this crew in gradio

In [ ]:
pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.9.4
    Uninstalling typer-0.9.4:
      Successfully uninstalled typer-0.9.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
instructor 0.5.2 requires typer<0.10.0,>=0.9.0, but you have typer 0.19.2 which is incompatible.


In [ ]:
import gradio as gr
from crewai import Agent, Task, Crew

# Define Agents
researcher = Agent(
    role="Senior Research Specialist",
    goal="Uncover intricate insights into the given topic",
    backstory="A seasoned expert in extracting valuable information and providing detailed analysis on any given subject.",
    allow_delegation=False
)

writer = Agent(
    role="Content Writer",
    goal="Craft engaging and informative articles",
    backstory="An experienced content writer who can turn complex research into easy-to-understand and engaging articles.",
    allow_delegation=False
)

editor = Agent(
    role="Content Editor",
    goal="Ensure articles are polished, coherent, and engaging",
    backstory="An expert editor with a sharp eye for clarity, grammar, and flow, capable of turning drafts into publication-ready articles.",
    allow_delegation=False
)

# Function to setup tasks and run CrewAI
def run_crewai_article_writer(topic):
    # Define tasks
    research_task = Task(
        description=f"Conduct in-depth research on the topic: {topic} and provide a structured summary.",
        expected_output="A structured and detailed research summary covering important aspects, recent developments, and insights related to the topic.",
        agent=researcher
    )

    writing_task = Task(
        description=f"Using the research, write a full-length article on the topic: {topic} in 1500 words give various sub section",
        expected_output="An engaging, detailed, informative, and coherent article suitable for online publication.",
        agent=writer
    )

    editing_task = Task(
        description=f"Review and edit the article on {topic} for clarity, grammar, flow, and engagement.",
        expected_output="A polished, clear, and professionally edited final version of the article.",
        agent=editor
    )

    # Setup Crew
    crew = Crew(
        agents=[researcher, writer, editor],
        tasks=[research_task, writing_task, editing_task],
        verbose=True  # Set to False if you want less console output
    )

    # Execute
    result = crew.kickoff()
    return result

# Gradio Interface
def generate_article(topic):
    return run_crewai_article_writer(topic)

iface = gr.Interface(
    fn=generate_article,
    inputs=gr.Textbox(lines=2, placeholder="Enter a topic for the article..."),
    outputs=gr.Textbox(lines=20, label="Generated Article"),
    title="AI Agent: Prompt to Post",
    description="Enter a topic and let CrewAI agents research, write, and edit a complete article!"
)

if __name__ == "__main__":
    iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7137c3a9004bc2e861.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
